# S - Scrub: Preprocessing dan Integrasi Data IDM 2024
## Kelompok 3 - II4013 Data Analytics

Notebook ini melakukan pembersihan data, konsolidasi data pendukung, integrasi dataset, rekayasa fitur (*feature engineering*), serta penandaan outlier.

### Alur Pengerjaan:
1. **Obtain**: Memuat dataset utama (`idm_2024_raw.csv`) dan dataset pendukung (32 file Excel rekomendasi provinsi).
2. **Clean Dataset Utama**: Menangani data non-aktif, mengisi data hilang, dan menstandarkan format tipe data.
3. **Konsolidasi Rekomendasi**: Membaca, ffill merged cells, dan menyatukan 32 berkas rekomendasi.
4. **Agregasi Rekomendasi**: Meringkas data rekomendasi dari relasi *one-to-many* menjadi *one-to-one* per desa.
5. **Integrasi Data**: Menggabungkan dataset utama dengan data rekomendasi teragregasi menggunakan `LEFT JOIN`.
6. **Feature Engineering**: Membuat fitur-fitur baru (`dimensi_terendah`, `gap_iks_ike`, `gap_iks_ikl`, `intensitas_rekomendasi`).
7. **Deteksi Outlier**: Menandai data outlier dengan batas IQR.
8. **Export**: Menyimpan hasil bersih ke `data/processed/idm_2024_modeling.csv`.

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import warnings
warnings.filterwarnings('ignore')

print('[OK] Library berhasil diimpor.')

In [ ]:
# Mendefinisikan base directory dan path data
BASE_DIR = os.path.dirname(os.getcwd())
RAW_DATA_PATH = os.path.join(BASE_DIR, 'data', 'raw', 'idm_2024_raw.csv')
REKOMENDASI_DIR = os.path.join(BASE_DIR, 'data', 'raw', 'rekomendasi_provinsi')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
PROCESSED_DATA_PATH = os.path.join(PROCESSED_DIR, 'idm_2024_modeling.csv')

print('Base Directory:', BASE_DIR)
print('Raw Data Path:', RAW_DATA_PATH)
print('Rekomendasi Dir:', REKOMENDASI_DIR)
print('Processed Data Path:', PROCESSED_DATA_PATH)

In [ ]:
# Memuat dataset utama
df_raw = pd.read_csv(RAW_DATA_PATH, low_memory=False)
print(f'Ukuran raw dataset utama: {df_raw.shape[0]:,} baris x {df_raw.shape[1]} kolom')
df_raw.head(3)

In [ ]:
# Mengidentifikasi 4 baris non-aktif yang memiliki keterangan teks di kolom IKS_2024
is_non_numeric = pd.to_numeric(df_raw['IKS_2024'], errors='coerce').isna()
df_anomali = df_raw[is_non_numeric]

print('--- Menampilkan 4 Desa Non-Aktif/Mengalami Anomali ---')
for idx, row in df_anomali.iterrows():
    print(f"Desa: {row['NAMA_DESA']} ({row['KODE_DESA']})")
    print(f"Alasan: {row['IKS_2024']}")
    print('-' * 50)

In [ ]:
# 1. Salin ke DataFrame baru untuk dibersihkan
df_clean = df_raw.copy()

# 2. Hapus baris desa yg non-aktif
df_clean = df_clean[~is_non_numeric].reset_index(drop=True)

# 3. Isi missing value NAMA_DESA untuk desa 1101012003
df_clean.loc[df_clean['KODE_DESA'] == 1101012003, 'NAMA_DESA'] = 'UJONG PADANG'

# 4. Format kolom KODE_DESA menjadi string 10 digit (untuk JOIN)
df_clean['KODE_DESA'] = df_clean['KODE_DESA'].astype(int).astype(str).str.strip()

# 5. Konversi kolom indeks komposit menjadi float64 secara aman
index_cols = ['IKS_2024', 'IKE_2024', 'IKL_2024', 'NILAI_IDM_2024']
for col in index_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    # Lakukan imputasi jika masih ada NaN (berdasarkan median sesuai instruksi)
    if df_clean[col].isna().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)

# 6. Tangani missing value pada status (modus)
if df_clean['STATUS_IDM_2024'].isna().sum() > 0:
    mode_status = df_clean['STATUS_IDM_2024'].mode()[0]
    df_clean['STATUS_IDM_2024'] = df_clean['STATUS_IDM_2024'].fillna(mode_status)

# 7. Buang kolom junk
cols_to_drop = [c for c in ['Keterangan', 'Unnamed: 14'] if c in df_clean.columns]
df_clean.drop(columns=cols_to_drop, inplace=True, errors='ignore')

print(f'Ukuran dataset utama setelah dibersihkan: {df_clean.shape[0]:,} baris x {df_clean.shape[1]} kolom')
print('Jumlah missing values di kolom inti:', df_clean[index_cols].isna().sum().sum())
df_clean.head(3)

In [ ]:
# Membaca dan menggabungkan 32 file Excel rekomendasi
rekom_files = glob.glob(os.path.join(REKOMENDASI_DIR, '*.xlsx'))
print(f'Menemukan {len(rekom_files)} berkas rekomendasi di folder raw.')

rekom_list = []
cols_to_fill = ['KODE PROV', 'NAMA PROVINSI', 'KODE KAB', 'NAMA KABUPATEN', 'KODE KEC', 'NAMA KECAMATAN', 'KODE DESA', 'NAMA DESA']

for file_path in rekom_files:
    try:
        # Load excel dengan header di baris ke-3 (baris ke-4 secara 1-indexed)
        df_file = pd.read_excel(file_path, header=3)
        
        # Standarisasi nama kolom: strip whitespace dan uppercase
        df_file.columns = df_file.columns.str.strip().str.upper()
        
        # Lakukan forward-fill pada kolom-kolom identitas karena cell merged
        fill_cols_exist = [c for c in cols_to_fill if c in df_file.columns]
        df_file[fill_cols_exist] = df_file[fill_cols_exist].ffill()
        
        rekom_list.append(df_file)
    except Exception as e:
        print(f"Gagal memproses file {os.path.basename(file_path)}: {e}")

df_rekom_all = pd.concat(rekom_list, ignore_index=True)
print(f'Total baris rekomendasi teronsolidasi: {len(df_rekom_all):,}')
df_rekom_all.head(3)

In [ ]:
# 1. Bersihkan kode desa rekomendasi
df_rekom_all['KODE DESA'] = df_rekom_all['KODE DESA'].dropna().astype(int).astype(str).str.strip()

# 2. Definisikan fungsi agregasi untuk relasi one-to-many
def join_unique_keterangan(series):
    unique_vals = series.dropna().unique()
    # Filter jika hanya berisi strip atau kosong
    unique_vals = [str(x).strip() for x in unique_vals if str(x).strip() not in ['', '-']]
    return '; '.join(unique_vals) if unique_vals else 'Tidak ada rekomendasi spesifik'

def get_mode_target(series):
    valid_series = series.dropna()
    if not valid_series.empty:
        return valid_series.mode()[0]
    return np.nan

df_rekom_agg = df_rekom_all.groupby('KODE DESA').agg(
    jumlah_rekomendasi=('KETERANGAN', 'count'),
    total_nilai_rekomendasi=('NILAI', 'sum'),
    keterangan_gabungan=('KETERANGAN', join_unique_keterangan),
    target_idm_2025=('TARGET IDM 2025', get_mode_target)
).reset_index()

print(f'Ukuran data rekomendasi setelah diagregasi per desa: {df_rekom_agg.shape[0]:,}')
df_rekom_agg.head(3)

In [ ]:
# Integrasi dataset menggunakan LEFT JOIN berdasarkan KODE_DESA
df_joined = df_clean.merge(df_rekom_agg, left_on='KODE_DESA', right_on='KODE DESA', how='left')

# Buat flag penanda ketersediaan data rekomendasi
df_joined['has_rekomendasi'] = df_joined['jumlah_rekomendasi'].notna()

# Mengisi nilai NaN pada desa yang tidak memiliki rekomendasi (misal provinsi yang absen)
df_joined['jumlah_rekomendasi'] = df_joined['jumlah_rekomendasi'].fillna(0).astype(int)
df_joined['total_nilai_rekomendasi'] = df_joined['total_nilai_rekomendasi'].fillna(0.0)
df_joined['keterangan_gabungan'] = df_joined['keterangan_gabungan'].fillna('Tidak ada data rekomendasi')
df_joined['target_idm_2025'] = df_joined['target_idm_2025'].fillna(df_joined['STATUS_IDM_2024'])

# Drop kolom KODE DESA duplikat hasil join
if 'KODE DESA' in df_joined.columns:
    df_joined.drop(columns=['KODE DESA'], inplace=True)

print(f'Ukuran dataset setelah digabungkan: {df_joined.shape[0]:,} baris x {df_joined.shape[1]} kolom')
print(f"Jumlah desa yang berhasil dipetakan ke rekomendasi: {df_joined['has_rekomendasi'].sum():,} dari {len(df_joined):,}")
df_joined.head(3)

In [ ]:
# Membuat minimal 3 fitur baru

# 1. dimensi_terendah: menentukan dimensi mana yang paling rendah skornya per desa
indeks_tiga = df_joined[['IKS_2024', 'IKE_2024', 'IKL_2024']]
df_joined['dimensi_terendah'] = indeks_tiga.idxmin(axis=1).map({
    'IKS_2024': 'Sosial',
    'IKE_2024': 'Ekonomi',
    'IKL_2024': 'Lingkungan'
})

# 2. gap_iks_ike: selisih nilai ketahanan sosial dan ketahanan ekonomi
df_joined['gap_iks_ike'] = df_joined['IKS_2024'] - df_joined['IKE_2024']

# 3. gap_iks_ikl: selisih nilai ketahanan sosial dan ketahanan lingkungan
df_joined['gap_iks_ikl'] = df_joined['IKS_2024'] - df_joined['IKL_2024']

# 4. intensitas_rekomendasi: jumlah rekomendasi dibagi rata-rata jumlah rekomendasi dari desa yang memiliki data rekomendasi
mean_rekom = df_joined[df_joined['has_rekomendasi']]['jumlah_rekomendasi'].mean()
df_joined['intensitas_rekomendasi'] = df_joined['jumlah_rekomendasi'] / (mean_rekom if mean_rekom > 0 else 1)

print('Fitur baru berhasil dibuat.')
print(df_joined[['dimensi_terendah', 'gap_iks_ike', 'gap_iks_ikl', 'intensitas_rekomendasi']].describe())
df_joined.head(3)

In [ ]:
# Deteksi outlier menggunakan metode IQR pada variabel numerik inti
def mark_outliers_iqr(df_in, col):
    q1 = df_in[col].quantile(0.25)
    q3 = df_in[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (df_in[col] < lower) | (df_in[col] > upper)

for col in ['IKS_2024', 'IKE_2024', 'IKL_2024', 'NILAI_IDM_2024']:
    outlier_col_name = f'is_outlier_{col.split("_")[0].lower()}'
    df_joined[outlier_col_name] = mark_outliers_iqr(df_joined, col)
    print(f"Jumlah outlier pada {col}: {df_joined[outlier_col_name].sum():,} ({df_joined[outlier_col_name].sum() / len(df_joined) * 100:.2f}%)")

print('Semua data outlier telah ditandai tanpa di-drop/di-cap.')

In [ ]:
# Membuat folder processed jika belum ada
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Ekspor data bersih ke file csv processed
df_joined.to_csv(PROCESSED_DATA_PATH, index=False, encoding='utf-8-sig')
print(f'[SUCCESS] File bersih disimpan ke: {PROCESSED_DATA_PATH}')
print(f'Jumlah akhir: {df_joined.shape[0]:,} baris x {df_joined.shape[1]} kolom')